# tuning

In [ ]:
# Import the experiment config, the results loader and the analysis functions.
import matplotlib.pyplot as plt
import numpy as np

from analysis.curves import ema, lifetime_average, return_curve, subsample
from analysis.stats import best_by_mean, mean_ci
from main import load_results

from config import EXPERIMENT

PLOTS_DIR = EXPERIMENT.results_dir.parent / "plots"

In [ ]:
# Load each agent's stored runs per environment and learning rate, if stored.
ENVIRONMENTS = [
    "cartpole",
    "acrobot",
    "mountaincar",
    "catch",
    "pinball_box",
    "pinball_easy",
    "pinball_empty",
    "pinball_medium",
]
CONTINUING = {"catch"}  # no episodes, so they are scored on reward
LEARNERS = ["dqn", "ddqn"]

results = {}
for environment in ENVIRONMENTS:
    runs = load_results(EXPERIMENT, f"random_{environment}")
    if runs is not None:
        results[environment, "random", None] = runs
    for learner in LEARNERS:
        name = f"{learner}_{environment}"
        for lr in EXPERIMENT.component(name).sweep["AGENT_HYPERS.LR"]:
            runs = load_results(EXPERIMENT, name, where={"AGENT_HYPERS.LR": lr})
            if runs is not None:
                results[environment, learner, lr] = runs

In [ ]:
# Compute each run's lifetime score, the best learning rate, and its learning curve.
def signal(environment, runs):
    if environment in CONTINUING:
        return runs.reward
    return return_curve(runs.reward, runs.done)


scores = {}
for (environment, agent, lr), runs in results.items():
    if lr is not None:
        scores.setdefault((environment, agent), {})[lr] = lifetime_average(
            signal(environment, runs)
        )
sensitivity = {
    key: {lr: mean_ci(runs) for lr, runs in by_lr.items()}
    for key, by_lr in scores.items()
}
best_lr = {key: best_by_mean(by_lr) for key, by_lr in scores.items()}
for (environment, agent), lr in best_lr.items():
    print(f"[{environment}] best {agent} LR: {lr}")

curves = {}
for (environment, agent, lr), runs in results.items():
    if lr is not None and best_lr.get((environment, agent)) != lr:
        continue
    raw = signal(environment, runs)
    smoothed = ema(raw) if environment in CONTINUING else raw
    timesteps, values = subsample(smoothed)
    curves[environment, agent] = (timesteps, mean_ci(values))

In [ ]:
# Style for the learning-rate sensitivity plots.
STYLE = {
    "figure.figsize": (9, 6),
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "lines.linewidth": 2.5,
    "legend.frameon": False,
}
LABELS = {"random": "Random", "dqn": "DQN", "ddqn": "DDQN"}
COLORS = {"random": "tab:red", "dqn": "tab:blue", "ddqn": "tab:green"}
BAND_ALPHA = 0.2
TITLES = {
    "cartpole": "Cartpole",
    "acrobot": "Acrobot",
    "mountaincar": "MountainCar",
    "catch": "Catch",
    "pinball_box": "Pinball (Box)",
    "pinball_easy": "Pinball (Easy)",
    "pinball_empty": "Pinball (Empty)",
    "pinball_medium": "Pinball (Medium)",
}
PLOT_TITLE = "{title} (10 seeds, 95% CI)"
YLABEL_KW = {"rotation": 0, "ha": "right", "va": "center", "labelpad": 12}
SENSITIVITY_LABEL = "{agent} (best LR={lr})"
SENSITIVITY_XLABEL = "Learning rate"
SENSITIVITY_XSCALE = "log"
SENSITIVITY_YLABELS = {
    "episodic": "Weighted\nlifetime\nreturn",
    "continuing": "Average\nlifetime\nreward",
}
SENSITIVITY_LEGEND = {"loc": "best"}


def kind(environment):
    return "continuing" if environment in CONTINUING else "episodic"

In [ ]:
# Plot each environment's mean lifetime score against learning rate.
figures = {}
for environment in ENVIRONMENTS:
    learners = [agent for agent in LEARNERS if (environment, agent) in sensitivity]
    if not learners:
        continue
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots()
        for agent in learners:
            lrs = sorted(sensitivity[environment, agent])
            intervals = [sensitivity[environment, agent][lr] for lr in lrs]
            mean, low, high = np.array(intervals).T
            color = COLORS[agent]
            ax.fill_between(lrs, low, high, color=color, alpha=BAND_ALPHA)
            label = SENSITIVITY_LABEL.format(
                agent=LABELS[agent], lr=best_lr[environment, agent]
            )
            ax.plot(lrs, mean, color=color, label=label)
        ax.set_xscale(SENSITIVITY_XSCALE)
        ax.set(title=PLOT_TITLE.format(title=TITLES[environment]))
        ax.set_xlabel(SENSITIVITY_XLABEL)
        ax.set_ylabel(SENSITIVITY_YLABELS[kind(environment)], **YLABEL_KW)
        ax.legend(**SENSITIVITY_LEGEND)
        fig.tight_layout()
    figures[f"{environment}_sensitivity"] = fig
plt.show()

In [ ]:
# Style for the best-learning-rate learning-curve plots.
YLIMS = {
    "cartpole": (0, 500),
    "acrobot": (-500, 0),
    "mountaincar": (-1000, 0),
    "catch": None,
    "pinball_box": (-1000, 0),
    "pinball_easy": (-1000, 0),
    "pinball_empty": (-1000, 0),
    "pinball_medium": (-1000, 0),
}
CURVE_XLABEL = "Timestep"
CURVE_YLABELS = {"episodic": "Return", "continuing": "Reward\n(EMA)"}
CURVE_LEGEND = {"loc": "lower right"}

In [ ]:
# Plot each environment's learning curves, each learner at its best learning rate.
for environment in ENVIRONMENTS:
    agents = [agent for agent in LABELS if (environment, agent) in curves]
    if not agents:
        continue
    with plt.rc_context(STYLE):
        fig, ax = plt.subplots()
        for agent in agents:
            timesteps, (mean, low, high) = curves[environment, agent]
            color = COLORS[agent]
            ax.fill_between(timesteps, low, high, color=color, alpha=BAND_ALPHA)
            ax.plot(timesteps, mean, color=color, label=LABELS[agent])
        ax.set(title=PLOT_TITLE.format(title=TITLES[environment]))
        ax.set(xlabel=CURVE_XLABEL, ylim=YLIMS[environment])
        ax.set_ylabel(CURVE_YLABELS[kind(environment)], **YLABEL_KW)
        ax.set_xticks([0, timesteps[-1]])  # timestep axis: show only 0 and the max
        ax.legend(**CURVE_LEGEND)
        fig.tight_layout()
    figures[f"{environment}_learning_curve"] = fig
plt.show()

In [ ]:
# Save every plotted figure as a PDF in the plots directory.
PLOTS_DIR.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(PLOTS_DIR / f"{name}.pdf", bbox_inches="tight")
print(f"saved {len(figures)} plot(s) to {PLOTS_DIR}")